# 🌾 MODULE 1: HUẤN LUYỆN MÔ HÌNH YOLO SEGMENTATION CHO HẠT LÚA
**Đề tài Nghiên cứu Khoa học**: Hệ thống thị giác máy tính và học sâu phục vụ ước lượng số lượng và đánh giá phẩm cấp hạt giống lúa.

### 🎯 Mục đích file này:
1. Kết nối Google Drive & cài đặt các thư viện cần thiết (`ultralytics`, `matplotlib`, `pyyaml`).
2. Cấu hình dataset và siêu tham số huấn luyện (Epochs, Batch size, Image size).
3. Huấn luyện mô hình phân đoạn cá thể hạt lúa (**YOLO Segmentation** - ví dụ `yolo26n-seg.pt` hoặc `yolov8n-seg.pt`).
4. Kiểm tra dự đoán thử nghiệm và tự động sao lưu toàn bộ trọng số (`best.pt`, `last.pt`) cùng biểu đồ mAP/Loss sang Google Drive.


In [ ]:
# Bước 1: Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Bước 2: Cài đặt thư viện Ultralytics YOLO
!pip install ultralytics matplotlib pyyaml


In [ ]:
# Bước 3: Import các thư viện cần thiết
import os
import shutil
import matplotlib.pyplot as plt
from ultralytics import YOLO


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Bước 4: CẤU HÌNH THAM SỐ HUẤN LUYỆN (Chỉnh sửa theo nhu cầu)
# ─────────────────────────────────────────────────────────────────────────────
# 1. Đường dẫn thư mục gốc trong Google Drive
BASE_PATH = "/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/"

# 2. Tên model pretrained và dataset
MODEL_NAME   = "yolo26n-seg.pt"  # Các lựa chọn: yolo26n-seg.pt, yolov8n-seg.pt, yolov11n-seg.pt
DATASET_NAME = "35_special_images_segmentation.v1i.yolov8"

# 3. Siêu tham số huấn luyện
EPOCHS     = 100
BATCH_SIZE = 16
IMG_SIZE   = 640
DEVICE     = 0  # Sử dụng GPU 0 (hoặc 'cpu' nếu không có GPU)

# 4. Tự động thiết lập đường dẫn lưu trữ
MODELS_DIR  = os.path.join(BASE_PATH, "MODELS")
DATASET_DIR = os.path.join(BASE_PATH, "DATASETS", DATASET_NAME)
DATA_YAML   = os.path.join(DATASET_DIR, "data.yaml")

TRAIN_EXP_NAME = f"{DATASET_NAME}_{MODEL_NAME.replace('.pt', '')}_trained"
DRIVE_SAVE_DIR = os.path.join(BASE_PATH, "RESULTS", TRAIN_EXP_NAME)

print("=" * 60)
print(f"📦 Model ban đầu     : {os.path.join(MODELS_DIR, MODEL_NAME)}")
print(f"📊 File data.yaml    : {DATA_YAML}")
print(f"💾 Nơi lưu kết quả   : {DRIVE_SAVE_DIR}")
print(f"⚙️ Cấu hình          : Epochs={EPOCHS} | Batch={BATCH_SIZE} | ImgSize={IMG_SIZE}")
print("=" * 60)


In [ ]:
# Bước 5: Bắt đầu Huấn luyện YOLO Segmentation
model_path = os.path.join(MODELS_DIR, MODEL_NAME)

if not os.path.exists(model_path):
    print(f"⚠️ Không tìm thấy model tại {model_path}. Sẽ tự động tải {MODEL_NAME} từ Ultralytics...")
    model = YOLO(MODEL_NAME)
else:
    model = YOLO(model_path)

print("🚀 Bắt đầu quá trình huấn luyện mô hình YOLO Segmentation...")
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    device=DEVICE,
    project="rice_seed_segmentation",
    name=TRAIN_EXP_NAME,
    exist_ok=True
)
print("✅ Quá trình huấn luyện đã hoàn tất!")


In [ ]:
# Bước 6: Thử nghiệm Dự đoán nhanh trên 1 ảnh mẫu
best_weight_local = f"/content/rice_seed_segmentation/{TRAIN_EXP_NAME}/weights/best.pt"

if os.path.exists(best_weight_local):
    trained_model = YOLO(best_weight_local)
    test_demo_url = "https://media.istockphoto.com/id/186786216/photo/unmilled-rice-grains.jpg"
    res = trained_model.predict(source=test_demo_url, conf=0.4, save=True)
    print("✅ Đã chạy thử nghiệm dự đoán mẫu thành công!")
else:
    print("⚠️ Không tìm thấy trọng số best.pt cục bộ.")


In [ ]:
# Bước 7: Tự động sao lưu toàn bộ kết quả vào Google Drive
local_result_dir = f"/content/rice_seed_segmentation/{TRAIN_EXP_NAME}"

if os.path.exists(local_result_dir):
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    shutil.copytree(local_result_dir, DRIVE_SAVE_DIR, dirs_exist_ok=True)
    print("=" * 60)
    print(f"🎉 ĐÃ SAO LƯU THÀNH CÔNG TOÀN BỘ KẾT QUẢ VÀO GOOGLE DRIVE!")
    print(f"📂 Thư mục: {DRIVE_SAVE_DIR}")
    print(f"⭐ Trọng số tốt nhất: {os.path.join(DRIVE_SAVE_DIR, 'weights', 'best.pt')}")
    print("=" * 60)
else:
    print("❌ Không tìm thấy thư mục kết quả để sao lưu.")


In [ ]:
# Bước 8: Hiển thị Biểu đồ Kết quả và Ma trận Nhầm lẫn
results_img_path = os.path.join(DRIVE_SAVE_DIR, "results.png")
confusion_matrix_path = os.path.join(DRIVE_SAVE_DIR, "confusion_matrix_normalized.png")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if os.path.exists(results_img_path):
    img_res = plt.imread(results_img_path)
    axes[0].imshow(img_res)
    axes[0].set_title("Quá trình Loss & mAP qua các Epochs", fontsize=12, fontweight='bold')
    axes[0].axis('off')

if os.path.exists(confusion_matrix_path):
    img_cm = plt.imread(confusion_matrix_path)
    axes[1].imshow(img_cm)
    axes[1].set_title("Ma trận nhầm lẫn chuẩn hóa (Confusion Matrix)", fontsize=12, fontweight='bold')
    axes[1].axis('off')

plt.tight_layout()
plt.show()
